# Reading & Writing Data

Data scientists work with data from many sources. This notebook covers reading from and writing to the most common formats:

- CSV (most common)
- JSON
- Excel
- SQLite / SQL databases
- Parquet (columnar format)
- Web (HTML tables, APIs)

In [ ]:
import pandas as pd
import numpy as np
import json
import os

# Helper — create a sample DataFrame we'll save and reload in multiple formats
np.random.seed(0)
df_orig = pd.DataFrame({
    'id':       range(1, 11),
    'name':     ['Alice','Bob','Carol','Dave','Eve','Frank','Grace','Hank','Ivy','Jack'],
    'dept':     ['Eng','Mkt','Eng','HR','Mkt','Eng','HR','Mkt','Eng','HR'],
    'salary':   np.random.randint(50000, 120000, 10),
    'joined':   pd.date_range('2020-01-01', periods=10, freq='3ME'),
    'active':   [True]*7 + [False]*3
})
print(df_orig)

## 1. CSV — Comma Separated Values

In [ ]:
# Write CSV
df_orig.to_csv('/tmp/employees.csv', index=False)
print('Saved CSV')

# Read CSV — most common params
df_csv = pd.read_csv('/tmp/employees.csv')
print(df_csv.dtypes)
print(df_csv.head(3))

In [ ]:
# Useful read_csv parameters
pd.read_csv('/tmp/employees.csv',
    usecols=['name','dept','salary'],   # only load these columns
    dtype={'salary': np.float32},       # override dtype
    parse_dates=['joined'],             # parse as datetime
    nrows=5,                            # read only first 5 rows
)

# Reading in chunks (for large files)
total_salary = 0
for chunk in pd.read_csv('/tmp/employees.csv', chunksize=3):
    total_salary += chunk['salary'].sum()
print('Total salary (chunked):', total_salary)

# Other delimiters
df_orig.to_csv('/tmp/employees.tsv', sep='\t', index=False)
df_tsv = pd.read_csv('/tmp/employees.tsv', sep='\t')
print(df_tsv.shape)

## 2. JSON

In [ ]:
# Write JSON
df_orig.to_json('/tmp/employees.json', orient='records', indent=2, date_format='iso')

# Read JSON
df_json = pd.read_json('/tmp/employees.json')
print(df_json.dtypes)
print(df_json.head(3))

# Nested JSON (API-style response)
import json

api_response = {
    'status': 'ok',
    'count': 3,
    'data': [
        {'id': 1, 'name': 'Alice', 'tags': ['python', 'ml']},
        {'id': 2, 'name': 'Bob',   'tags': ['sql', 'bi']},
        {'id': 3, 'name': 'Carol', 'tags': ['python', 'stats']}
    ]
}

# Normalise nested JSON into a flat table
df_api = pd.json_normalize(api_response['data'])
print(df_api)

## 3. Excel

In [ ]:
# Write Excel (requires openpyxl: uv add openpyxl)
try:
    df_orig.to_excel('/tmp/employees.xlsx', sheet_name='Employees', index=False)

    # Read Excel
    df_excel = pd.read_excel('/tmp/employees.xlsx', sheet_name='Employees')
    print(df_excel.head(3))

    # Write multiple sheets
    with pd.ExcelWriter('/tmp/multi_sheet.xlsx', engine='openpyxl') as writer:
        df_orig[df_orig['dept']=='Eng'].to_excel(writer, sheet_name='Engineering', index=False)
        df_orig[df_orig['dept']=='HR'].to_excel(writer,  sheet_name='HR', index=False)

    print('Multi-sheet Excel written')
except ImportError:
    print('Install openpyxl: uv add openpyxl')

## 4. SQLite

In [ ]:
import sqlite3

# Write DataFrame directly to SQLite
conn = sqlite3.connect('/tmp/company.db')
df_orig.to_sql('employees', conn, if_exists='replace', index=False)
print('Written to SQLite')

# Read with SQL query
df_sql = pd.read_sql('SELECT * FROM employees WHERE dept = "Eng"', conn)
print(df_sql)
conn.close()

In [ ]:
# More complex queries
conn = sqlite3.connect('/tmp/company.db')

df_agg = pd.read_sql('''
    SELECT dept,
           COUNT(*)         AS headcount,
           AVG(salary)      AS avg_salary,
           MAX(salary)      AS max_salary
    FROM   employees
    GROUP  BY dept
    ORDER  BY avg_salary DESC
''', conn)
print(df_agg)
conn.close()

## 5. Parquet — High-Performance Columnar Format

Parquet is the preferred format for large datasets — much faster to read/write than CSV, preserves dtypes.

In [ ]:
# Requires pyarrow: uv add pyarrow
try:
    df_orig.to_parquet('/tmp/employees.parquet', index=False)
    df_parquet = pd.read_parquet('/tmp/employees.parquet')
    print(df_parquet.dtypes)  # dtypes are preserved!
    print(df_parquet.head(3))
except ImportError:
    print('Install pyarrow: uv add pyarrow')

## 6. Reading HTML Tables from the Web

In [ ]:
# pd.read_html() extracts all <table> elements from an HTML page or string
sample_html = '''
<table>
  <thead><tr><th>Country</th><th>GDP_bn</th><th>Population_m</th></tr></thead>
  <tbody>
    <tr><td>USA</td><td>25000</td><td>335</td></tr>
    <tr><td>China</td><td>18000</td><td>1412</td></tr>
    <tr><td>Germany</td><td>4200</td><td>84</td></tr>
  </tbody>
</table>
'''

tables = pd.read_html(sample_html)
df_html = tables[0]
print(df_html)

# To read from a real URL (needs internet):
# tables = pd.read_html('https://en.wikipedia.org/wiki/List_of_countries_by_GDP')
# print(len(tables), 'tables found')

## 7. Saving Data — Best Practices

| Format | Use when | Pros | Cons |
|--------|----------|------|------|
| CSV | Sharing, interop | Universal | Slow, no dtype preservation |
| JSON | APIs, nested data | Human-readable | Slow for large data |
| Excel | Non-technical users | Familiar | Slow, large files |
| Parquet | Big data, pipelines | Fast, small, preserves dtypes | Not human-readable |
| SQLite | Relational queries | SQL support | Single-file DB only |

In [ ]:
# Performance comparison: CSV vs Parquet
import time

# Create a larger DataFrame
big_df = pd.DataFrame(np.random.randn(100_000, 10), columns=list('ABCDEFGHIJ'))

# CSV write/read time
start = time.perf_counter()
big_df.to_csv('/tmp/big.csv', index=False)
t_csv_write = time.perf_counter() - start

start = time.perf_counter()
pd.read_csv('/tmp/big.csv')
t_csv_read = time.perf_counter() - start

# Parquet write/read time
try:
    start = time.perf_counter()
    big_df.to_parquet('/tmp/big.parquet', index=False)
    t_parq_write = time.perf_counter() - start

    start = time.perf_counter()
    pd.read_parquet('/tmp/big.parquet')
    t_parq_read = time.perf_counter() - start

    print(f'CSV   write: {t_csv_write:.3f}s  |  read: {t_csv_read:.3f}s')
    print(f'Parquet write: {t_parq_write:.3f}s  |  read: {t_parq_read:.3f}s')
except ImportError:
    print(f'CSV write: {t_csv_write:.3f}s  read: {t_csv_read:.3f}s')
    print('Install pyarrow for Parquet comparison')

## Quick Summary

| Format | Read | Write |
|--------|------|-------|
| CSV | `pd.read_csv('file.csv')` | `df.to_csv('file.csv', index=False)` |
| JSON | `pd.read_json('file.json')` | `df.to_json('file.json', orient='records')` |
| Excel | `pd.read_excel('file.xlsx')` | `df.to_excel('file.xlsx', index=False)` |
| SQLite | `pd.read_sql('SELECT ...', conn)` | `df.to_sql('table', conn)` |
| Parquet | `pd.read_parquet('file.parquet')` | `df.to_parquet('file.parquet')` |
| HTML | `pd.read_html(url)` | — |

**Next →** [05 – Matplotlib](../05-matplotlib/)